In [1]:
from pathlib import Path
import sys

# Ensure project root is on sys.path for imports
project_root = Path("../..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dataloader import load_txt, preprocess, GWASDataset

data_path = Path("../../data/tmpDATA-Leon/donnees_MRI_MDD_only_variants_clumping_p_thr_0.001all.txt")

df = load_txt(data_path)
df.head(5)

,Mean_intensity_3rd-Ventricle_whole-brain,Mean_intensity_4th-Ventricle_whole-brain,Mean_intensity_Brain-Stem_whole-brain,Mean_intensity_CSF_whole-brain,Mean_intensity_WM-hypointensities_whole-brain,Mean_intensity_Optic-Chiasm_whole-brain,Mean_intensity_CC-Posterior_whole-brain,Mean_intensity_CC-Mid-Posterior_whole-brain,Mean_intensity_CC-Central_whole-brain,Mean_intensity_CC-Mid-Anterior_whole-brain,...,Volume_S-postcentral_right,Volume_S-precentral-inf-part_right,Volume_S-precentral-sup-part_right,Volume_S-suborbital_right,Volume_S-subparietal_right,Volume_S-temporal-inf_right,Volume_S-temporal-sup_right,Volume_S-temporal-transverse_right,Z_scores_MDD,ID
0,-0.909895,-0.335305,0.519297,-0.555432,0.756546,-0.477407,1.212950,0.529090,1.019100,1.192580,...,-0.603143,-0.118828,-1.353370,1.423650,-0.934246,-0.106361,-0.201944,0.219648,-3.390360,rs10903385
1,-0.678412,1.166110,-0.162587,-1.929640,0.855909,-0.143850,-1.085770,0.610258,1.297300,0.702451,...,0.914586,-1.024000,0.280131,2.874470,0.694286,0.309240,-0.533916,1.673910,-3.311167,rs1392828
2,1.116720,0.719322,2.042310,1.862870,-1.313900,-0.488740,0.178764,0.298260,0.458461,-0.058223,...,-0.883645,0.157822,-0.976112,-1.539300,0.775611,-0.931109,-0.822349,-0.844325,3.633392,rs145358842
3,-0.039337,1.658240,0.327507,0.895273,1.092190,0.191609,0.310098,1.277860,0.387303,-1.085630,...,1.723490,0.557382,1.783520,-0.308143,-1.684860,-0.497441,-0.534644,1.414330,-3.835441,rs188848422
4,0.494151,1.168270,0.949132,0.897002,2.057640,0.762612,0.490406,-0.150986,1.059360,-1.107590,...,0.841207,1.862120,1.522980,0.876647,-0.073244,-0.276852,0.094010,1.677220,-3.432097,rs10903459


In [2]:
X_train, y_train, X_test, y_test = preprocess(df=df, target="Z_scores_MDD", testsize = 0.2)

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# Load Boston Housing data
#df = fetch_openml(data_id=531, as_frame=True)  # Boston Housing dataset
#X = df.data
#y = df.target.astype(float)  # Ensure target is float for regression
#
## Train-test split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Initialize the regressor
regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor.fit(X_train, y_train)

# Predict on the test set
predictions = regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Squared Error (MSE):", mse)
print("R² Score:", r2)

TabPFNMPSOutOfMemoryError: MPS out of memory with 809 test samples.

Solution: Split your test data into smaller batches:

    batch_size = 1000  # depends on hardware
    predictions = []
    for i in range(0, len(X_test), batch_size):
        batch = model.predict(X_test[i:i + batch_size])
        predictions.append(batch)
    predictions = np.vstack(predictions)

Original error: MPS backend out of memory (MPS allocated: 3.65 GiB, other allocations: 4.12 GiB, max allowed: 8.40 GiB). Tried to allocate 2.56 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).